# Retrain AntMaze Checkpoint

Notebook này dùng cho Kaggle để retrain AntMaze theo curriculum: bắt đầu với `AntMazeUMaze`, checkpoint định kỳ, resume sau timeout, kiểm tra readiness, rồi upload checkpoint lên Hugging Face Dataset repo.

Không hard-code token trong notebook. Trên Kaggle, thêm secret `HF_TOKEN` nếu muốn download/upload checkpoint.

## Configuration

`AntMazeUMaze` là bước đầu tiên. Chỉ chuyển sang `AntMazeMedium` sau khi readiness của U-Maze đạt baseline clean đủ cao.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Jun1801/latent_landmarks.git"
BRANCH = "feature/mcts-landmark"
REPO_DIR = Path("/kaggle/working/latent_landmarks")

# Main retrain target. Start here before AntMazeMedium/Large.
TRAIN_ENV = "AntMazeUMaze"
TARGET_STEPS = 3_000_000
WORKERS = 3
SAVE_EVERY = 200_000
EVAL_EPISODES = 10
SEED = 0

CHECKPOINT_NAME = "l3p_ant_umaze.pt"
OUTPUT_DIR = Path("/kaggle/working/antmaze_retrain_outputs")
SAVE_PATH = Path("checkpoint") / CHECKPOINT_NAME
LOG_PATH = Path("logs") / f"{TRAIN_ENV}.log"
PLOT_PATH = OUTPUT_DIR / f"{TRAIN_ENV}_learning_curve.png"
READINESS_PATH = OUTPUT_DIR / f"{TRAIN_ENV}_readiness.json"

# Hugging Face Dataset repo for ignored .pt checkpoints.
HF_REPO_ID = "Jun1801/mcts_vla"
DOWNLOAD_EXISTING_CHECKPOINT = True
UPLOAD_CHECKPOINT_TO_HF = True

# Optional after readiness passes.
RUN_E1_SMOKE_IF_READY = False
MIN_BASELINE_SUCCESS = 0.30

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configured", TRAIN_ENV, "target_steps=", TARGET_STEPS)

## Load Secrets

Cell này đọc `HF_TOKEN` từ Kaggle Secrets nếu có. Token chỉ được set vào environment, không in ra output.

In [ ]:
def load_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
        return token
    except Exception:
        return None

HF_TOKEN = load_hf_token()
print("HF token available:", bool(HF_TOKEN))

## Clone Repo

Clone đúng branch `feature/mcts-landmark`. Nếu repo đã tồn tại, cell này fetch và pull fast-forward.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("cwd:", Path.cwd())
subprocess.run(["git", "branch", "--show-current"], check=True)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

## Install Dependencies

AntMaze cần MuJoCo và Gymnasium-Robotics. Cell này pin `gymnasium-robotics==1.4.2`, phiên bản đã được smoke test với repo này.

In [ ]:
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / "mplconfig"))
os.environ.setdefault("XDG_CACHE_HOME", str(OUTPUT_DIR / ".cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "mujoco>=3", "gymnasium-robotics==1.4.2", "huggingface_hub"
], check=True)

print("Dependencies ready.")

## Smoke Test Environment

Cell này chỉ tạo env và đọc dimensions. Nếu lỗi ở đây thì chưa nên train.

In [ ]:
from l3p.config import get_config, env_spec
from l3p.envs import make_vec_env

cfg = get_config(TRAIN_ENV, seed=SEED, n_workers=1)
venv = make_vec_env(cfg, 1, cfg.seed)
obs = venv.envs[0].reset()
print("env:", TRAIN_ENV, env_spec(TRAIN_ENV))
print("obs_dim", venv.obs_dim, "goal_dim", venv.goal_dim, "act_dim", venv.act_dim)
print("observation keys:", sorted(obs.keys()))

## Optional: Download Existing Checkpoint

Dùng khi muốn resume từ checkpoint đã upload lên Hugging Face. Nếu repo private, cần `HF_TOKEN`.

In [ ]:
Path("checkpoint").mkdir(exist_ok=True)
if DOWNLOAD_EXISTING_CHECKPOINT and HF_REPO_ID:
    cmd = [
        sys.executable, "scripts/hf_sync_checkpoints.py", "download",
        "--repo-id", HF_REPO_ID,
        "--checkpoint-dir", "checkpoint",
        "--allow-patterns", CHECKPOINT_NAME, f"**/{CHECKPOINT_NAME}"
    ]
    if HF_TOKEN:
        cmd += ["--token", HF_TOKEN]
    print("$", " ".join(cmd[:-1] + (["***"] if HF_TOKEN else [])))
    rc = subprocess.run(cmd).returncode
    print("download returncode:", rc)
else:
    print("Checkpoint download skipped.")

print("existing checkpoint:", SAVE_PATH.exists(), SAVE_PATH)

## Train Or Resume

Nếu `checkpoint/l3p_ant_umaze.pt` đã tồn tại, command sẽ resume. `--steps` là tổng target absolute; ví dụ checkpoint đang 1M và `TARGET_STEPS=3M` thì train tiếp tới 3M.

In [ ]:
cmd = [
    sys.executable, "scripts/train.py",
    "--env", TRAIN_ENV,
    "--steps", str(TARGET_STEPS),
    "--seed", str(SEED),
    "--workers", str(WORKERS),
    "--save", str(SAVE_PATH),
    "--save-every", str(SAVE_EVERY),
    "--log-file", str(LOG_PATH),
    "--eval-episodes", str(EVAL_EPISODES),
]
if SAVE_PATH.exists():
    cmd += ["--resume", str(SAVE_PATH)]

print("$", " ".join(map(str, cmd)))
subprocess.run(cmd, check=True, env=os.environ.copy())

## Plot Training Curve

Vẽ learning curve từ log nếu đã có eval lines.

In [ ]:
if LOG_PATH.exists():
    cmd = [
        sys.executable, "scripts/plot_log.py",
        "--log", str(LOG_PATH),
        "--out", str(PLOT_PATH),
        "--title", f"L3P on {TRAIN_ENV}",
    ]
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=False, env=os.environ.copy())
else:
    print("No log file yet:", LOG_PATH)

## Readiness Check

Chỉ chạy E1a/E1b khi clean calibrated Soft Floyd baseline đạt ngưỡng. Với AntMaze, nếu flat/planner vẫn 0 thì cần train tiếp hoặc quay lại U-Maze.

In [ ]:
ready = False
if SAVE_PATH.exists():
    cmd = [
        sys.executable, "scripts/check_checkpoint_readiness.py",
        "--env", TRAIN_ENV,
        "--load", str(SAVE_PATH),
        "--episodes", "20",
        "--calibrate-episodes", "10",
        "--min-success", str(MIN_BASELINE_SUCCESS),
        "--out", str(READINESS_PATH),
    ]
    print("$", " ".join(map(str, cmd)))
    rc = subprocess.run(cmd, check=False, env=os.environ.copy()).returncode
    ready = (rc == 0)
    print("READY:", ready, "returncode:", rc)
else:
    print("No checkpoint found:", SAVE_PATH)

## Optional E1 Smoke After READY

Mặc định tắt. Chỉ bật khi readiness pass; mục tiêu là sanity nhanh, không phải report.

In [ ]:
if RUN_E1_SMOKE_IF_READY and ready:
    manifest_path = OUTPUT_DIR / "antmaze_retrain_e1_manifest.json"
    manifest = [{
        "name": "antmaze_retrain",
        "env": TRAIN_ENV,
        "checkpoint": str(SAVE_PATH),
        "experiments": ["e1a", "e1b"],
    }]
    manifest_path.write_text(json.dumps(manifest, indent=2))
    cmd = [
        sys.executable, "scripts/kaggle_run_e1_all.py",
        "--preset", "smoke",
        "--manifest", str(manifest_path),
        "--only", "antmaze_retrain",
        "--experiments", "e1a", "e1b",
        "--require-ready",
        "--min-baseline-success", str(MIN_BASELINE_SUCCESS),
        "--output-root", str(OUTPUT_DIR / "e1_smoke"),
    ]
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=False, env=os.environ.copy())
else:
    print("E1 smoke skipped. RUN_E1_SMOKE_IF_READY=", RUN_E1_SMOKE_IF_READY, "READY=", ready)

## Collect Outputs

Copy checkpoint/log/readiness/plot vào `/kaggle/working/antmaze_retrain_outputs` để tải về từ Kaggle.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for p in [SAVE_PATH, LOG_PATH, PLOT_PATH, READINESS_PATH]:
    p = Path(p)
    if p.exists() and p.is_file():
        dst = OUTPUT_DIR / p.name
        if p.resolve() != dst.resolve():
            shutil.copy2(p, dst)
        print("copied", p, "->", dst)

print("Outputs:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" ", p, p.stat().st_size, "bytes")

## Upload Checkpoint To Hugging Face

Upload chỉ checkpoint AntMaze retrain, không upload toàn bộ `checkpoint/`.

In [ ]:
if UPLOAD_CHECKPOINT_TO_HF and HF_REPO_ID and HF_TOKEN and SAVE_PATH.exists():
    upload_dir = OUTPUT_DIR / "hf_upload"
    if upload_dir.exists():
        shutil.rmtree(upload_dir)
    upload_dir.mkdir(parents=True)
    shutil.copy2(SAVE_PATH, upload_dir / SAVE_PATH.name)
    cmd = [
        sys.executable, "scripts/hf_sync_checkpoints.py", "upload",
        "--repo-id", HF_REPO_ID,
        "--checkpoint-dir", str(upload_dir),
        "--token", HF_TOKEN,
        "--create-repo",
    ]
    safe_cmd = list(cmd)
    safe_cmd[safe_cmd.index("--token") + 1] = "***"
    print("$", " ".join(map(str, safe_cmd)))
    subprocess.run(cmd, check=True)
else:
    print("HF upload skipped.")
    print("UPLOAD_CHECKPOINT_TO_HF=", UPLOAD_CHECKPOINT_TO_HF, "HF_REPO_ID=", bool(HF_REPO_ID), "HF_TOKEN=", bool(HF_TOKEN), "checkpoint=", SAVE_PATH.exists())

## Next Step: Warm-Start Medium

Chỉ chạy cell này sau khi `AntMazeUMaze` readiness đạt ngưỡng. `--reset-counters` để train Medium từ 0 steps nhưng dùng weight U-Maze làm init.

In [ ]:
RUN_MEDIUM_WARM_START = False
if RUN_MEDIUM_WARM_START:
    medium_ckpt = Path("checkpoint/l3p_ant_medium.pt")
    medium_log = Path("logs/AntMazeMedium.log")
    cmd = [
        sys.executable, "scripts/train.py",
        "--env", "AntMazeMedium",
        "--steps", "3000000",
        "--workers", str(WORKERS),
        "--resume", str(SAVE_PATH),
        "--reset-counters",
        "--save", str(medium_ckpt),
        "--save-every", str(SAVE_EVERY),
        "--log-file", str(medium_log),
        "--eval-episodes", str(EVAL_EPISODES),
    ]
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, env=os.environ.copy())
else:
    print("Medium warm-start disabled.")